In [10]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed
import pyart
import geopy.distance as distance
import haversine

In [11]:
#print(P2)

In [12]:
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100m.nc')
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP4.nc')
ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mnew.nc')


In [13]:
location = ncfile1.variables['location'][:]
qc = ncfile1.variables['qc'][:]
obstype = ncfile1.variables['obs_type'][:]
obstypemd = ncfile1.variables['ObsTypesMetaData'][:]
obs_val = ncfile1.variables['observations'][:]
which_vert = ncfile1.variables['which_vert'][:]
times = ncfile1.variables['time'][:]
print(obstype)
qc_new = []
for i in range(len(qc)):
    qc_d = qc[i][0]
    qc_new.append(qc_d)
qc_new = np.asarray(qc_new)

[142 105 106 ...  26  27  28]


In [ ]:
otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []

minute_range = np.arange(180,725,5)
#minute_range = np.arange(350,365,5)
#minute_range = np.arange(595,605,5)

for mins in minute_range:
    #dt_start = datetime(2022,9,15,0,0)
    #dt_start = datetime(2022,7,19,0,0)
    dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    print(dt)
    #dt = datetime(2022,9,15,9,55)
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    
    
    #otype = 107
    #otype = 105
    #otype = 106
    #otype = 108
    #otype = 142
    otype_list = [107,105,106,108]
    #otype_list = [107]
    for otype in otype_list:
        loc_T2 = location[obstype==otype, :]
        qc_T2 = qc_new[obstype==otype]
        obs_T2 = obs_val[obstype==otype, :]
        lons_T2 = loc_T2[:,0]
        lats_T2 = loc_T2[:,1]
        elev_T2 = loc_T2[:,2]
        time_T2 = times[obstype==otype]
        lons_T2[lons_T2 > 180] = lons_T2[lons_T2 > 180] - 360
        
        #Convert WRF file time into same units as the obs_seq time
        dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400
        time_diff = np.abs(dt_tot - time_T2)
        
        #Get obs within +- 2.5 minutes of each WRF file
        time_T3 = time_T2[time_diff<(150/86400)]
        loc_T3 = loc_T2[time_diff<(150/86400), :]
        qc_T3 = qc_T2[time_diff<(150/86400)]
        lons_T3 = lons_T2[time_diff<(150/86400)]
        lats_T3 = lats_T2[time_diff<(150/86400)]
        elev_T3 = elev_T2[time_diff<(150/86400)]
        obs_T3 = obs_T2[time_diff<(150/86400), :]
        
        if len(obs_T3)==0:
            print('NO OBS IN WINDOW')
        for k in range(len(lons_T3)):
            latp=lats_T3[k]
            lonp=lons_T3[k]
            #Get location for each ob in model land
            lon1d = np.ndarray.flatten(lon[0,:,:])
            lat1d = np.ndarray.flatten(lat[0,:,:])
            station = []
            points = []
            for i in range(len(lon1d)):
                points.append((lat1d[i],lon1d[i]))
                station.append((latp,lonp))
            dist = haversine.haversine_vector(station,points)
            dist2=dist.reshape(lon.shape[1],lon.shape[2])
            print(lon[0,:,:][np.where(dist2==np.min(dist2))])
            print(lat[0,:,:][np.where(dist2==np.min(dist2))])
            print(np.where(dist2==np.min(dist2)))
            st_xind = np.where(dist2==np.min(dist2))[0][0]
            st_yind = np.where(dist2==np.min(dist2))[1][0]
            if otype == 107:
                T2_a = T2[0,st_xind,st_yind].magnitude
            elif otype == 105:
                T2_a = U10[0,st_xind,st_yind]
            elif otype == 106:
                T2_a = V10[0,st_xind,st_yind]
            elif otype == 108:
                T2_a = Q2[0,st_xind,st_yind]
            #If you want to change the error assumption, just change the scale in this line
            error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[k,1]))
            #6/17/2024 MW adding code to limit added error to 1.5 times the error assumtion in DART
            if np.abs(error/4) > (np.sqrt(obs_T3[k,1])*1.0):
                error = (error / np.abs(error)) * (np.sqrt(obs_T3[k,1])*1.0)
            T2_b = T2_a + (error/4)
            print(T2_a, (error/4))
        
            #Append obs to arrays for writing to obs_seq file later
            otype_s.append(otype)
            obs_s.append(T2_b)
            lon_s.append(lonp)
            lat_s.append(latp)
            elev_s.append(elev_T3[k])
            error_s.append(obs_T3[k,1]/2)
            time_s.append(time_T3[k])

2021-06-04 03:00:00
NO OBS IN WINDOW
NO OBS IN WINDOW
NO OBS IN WINDOW
NO OBS IN WINDOW
2021-06-04 03:05:00
NO OBS IN WINDOW
NO OBS IN WINDOW
NO OBS IN WINDOW
NO OBS IN WINDOW
2021-06-04 03:10:00
NO OBS IN WINDOW
NO OBS IN WINDOW
NO OBS IN WINDOW
NO OBS IN WINDOW
2021-06-04 03:15:00
[-84.77993]
[39.24957]
(array([579]), array([505]))
289.42328 -0.2219187613510023
[-84.21958]
[39.08033]
(array([413]), array([941]))
288.0391 -0.8562057703812836
[-85.260086]
[39.350243]
(array([679]), array([133]))
291.0491 0.6747567446901479
[-84.39989]
[39.530434]
(array([862]), array([797]))
289.23865 1.2081456084571924
[-84.77993]
[39.24957]
(array([579]), array([505]))
0.40557235 -0.44217357996323225
[-84.21958]
[39.08033]
(array([413]), array([941]))
1.8249636 -0.4564571057668349
[-85.260086]
[39.350243]
(array([679]), array([133]))
0.17850208 -1.1436394940573744
[-84.25053]
[39.460365]
(array([793]), array([913]))
1.1525989 1.364626513299598
[-84.39989]
[39.530434]
(array([862]), array([797]))
0.63

In [ ]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
#day_DART = 154024
#day_DART = 153966
day_DART = 153556
seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [ ]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
print(seconds_DART)
print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]

In [ ]:
for bigfoot in [1,2]:
    print(bigfoot)

    #Write the simulated obs out to an obs_seq file
    #filename = 'SIM_METAR_IOP6_finalhalferr'
    #filename = 'SIM_METAR_IOP4_finalhalferr'
    filename = 'SIM_METAR_JUNE_finalhalferr'
    #filename = 'SIM_METAR_IOP6_finaloneob'
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(1))
    fi.write("    %d          %s   \n" %(107, 'METAR_TEMPERATURE_2_METER'))
    fi.write("    %d          %s   \n" %(105, 'METAR_U_10_METER_WIND'))
    fi.write("    %d          %s   \n" %(106, 'METAR_V_10_METER_WIND'))
    fi.write("    %d          %s   \n" %(108, 'METAR_SPECIFIC_HUMIDITY_2_METER'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s1)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s1):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], elev_s1[q], -1))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

In [ ]:
print(obs_T3[:,1])

#Get a randomly-generated error value to add to the synthetic ob 
#by sampling a Gaussian distribution with the same standard deviation as
#the assumed error from DART
error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[0,1]))
print(error)

In [ ]:
print(len(obs_s))

In [ ]:
#Convert WRF file time into same units as the obs_seq time
dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400

time_diff = np.abs(dt_tot - time_T2)

print(time_diff[time_diff<(150/86400)])

print(dt_tot)
print(time_diff)

In [ ]:
print(dt.isoformat()[14:16])

In [ ]:
print(len(obs_s))